In [1]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/Superstore.csv", encoding="latin1")

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

In [3]:
df["Delivery Days"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

In [4]:
df[["Order Date","Ship Date","Delivery Days"]].head()

,Order Date,Ship Date,Delivery Days
0,2016-11-08,2016-11-11,3
1,2016-11-08,2016-11-11,3
2,2016-06-12,2016-06-16,4
3,2015-10-11,2015-10-18,7
4,2015-10-11,2015-10-18,7


In [5]:
df["Profit Margin"] = (
    df["Profit"] / df["Sales"]
)

In [6]:
df["Profit Margin"].describe()

count    9994.000000
mean        0.120314
std         0.466754
min        -2.750000
25%         0.075000
50%         0.270000
75%         0.362500
max         0.500000
Name: Profit Margin, dtype: float64

In [7]:
df["Discount Percentage"] = df["Discount"] * 100

In [8]:
df["Demand Score"] = (
    df["Quantity"] * df["Sales"]
)

In [9]:
import numpy as np

np.random.seed(42)

df["Stock Level"] = np.random.randint(
    20,
    300,
    size=len(df)
)

In [10]:
np.random.seed(42)

variation = np.random.uniform(
    -0.10,
    0.10,
    len(df)
)

df["Competitor Price"] = (
    df["Sales"] * (1 + variation)
).round(2)

In [11]:
weather = [
    "Sunny",
    "Rainy",
    "Cloudy"
]

df["Weather"] = np.random.choice(
    weather,
    size=len(df)
)

In [12]:
df["Holiday"] = np.random.choice(
    [0,1],
    size=len(df),
    p=[0.9,0.1]
)

In [13]:
df["Weekend"] = (
    df["Order Date"].dt.weekday >= 5
).astype(int)

In [20]:
df.to_csv(
    "../data/processed/dynamic_pricing_dataset.csv",
    index=False
)

In [17]:
import os

print(os.getcwd())

c:\Users\uttra\OneDrive\Desktop\help!\Dynamic Pricing Optimization\notebooks


In [18]:
import os

print(os.path.exists("../data/processed"))

False


In [19]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/dynamic_pricing_dataset.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [21]:
df = pd.read_csv(
    "../data/processed/dynamic_pricing_dataset.csv"
)

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Delivery Days,Profit Margin,Discount Percentage,Demand Score,Stock Level,Competitor Price,Weather,Holiday,Weekend,Demand Level,Recommended Price
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,3,0.1600,0.0,523.9200,122,255.39,Sunny,1,0,High,284.10
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,3,0.3000,0.0,2195.8200,290,797.92,Sunny,0,0,High,861.75
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,4,0.4700,0.0,29.2400,126,15.30,Rainy,0,1,Low,14.54
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,7,-0.4000,45.0,4787.8875,91,976.47,Sunny,0,1,High,935.95
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,7,0.1125,20.0,44.7360,208,20.83,Rainy,0,1,Low,18.80


In [22]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Delivery Days', 'Profit Margin', 'Discount Percentage', 'Demand Score', 'Stock Level', 'Competitor Price', 'Weather', 'Holiday', 'Weekend', 'Demand Level', 'Recommended Price'], dtype='str')

In [23]:
# Create Demand Level

df["Demand Level"] = pd.qcut(
    df["Demand Score"],
    q=3,
    labels=["Low", "Medium", "High"]
)

df["Demand Level"].value_counts()

Demand Level
Low       3332
Medium    3331
High      3331
Name: count, dtype: int64

In [24]:
def recommend_price(row):

    price = row["Competitor Price"]

    # Demand Adjustment
    if row["Demand Level"] == "High":
        price *= 1.08
    elif row["Demand Level"] == "Low":
        price *= 0.95

    # Holiday Adjustment
    if row["Holiday"] == 1:
        price *= 1.03

    # Stock Adjustment
    if row["Stock Level"] < 50:
        price *= 1.05

    # Discount Adjustment
    price *= (1 - row["Discount"] * 0.25)

    return round(price, 2)

In [25]:
df["Recommended Price"] = df.apply(
    recommend_price,
    axis=1
)

df[[
    "Sales",
    "Competitor Price",
    "Recommended Price"
]].head()

,Sales,Competitor Price,Recommended Price
0,261.9600,255.39,284.10
1,731.9400,797.92,861.75
2,14.6200,15.30,14.54
3,957.5775,976.47,935.95
4,22.3680,20.83,18.80


In [26]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/dynamic_pricing_dataset.csv",
    index=False
)

print("Final engineered dataset saved!")

Final engineered dataset saved!
